# Finetuning - QLoRA

https://huggingface.co/docs/peft/en/developer_guides/quantization


## Quantization
4비트 양자화(4-bit Quantization)는 모델의 가중치를 정밀도가 낮은 4비트 데이터 형식으로 변환하여 메모리 사용량을 획기적으로 줄이는 기술이다. 지적한 대로 모든 파라미터가 양자화 대상이 되는 것은 아니며, 성능 유지를 위해 전략적으로 적용된다.
4비트 양자화는 **"대세인 가중치는 작게 줄이고, 민감한 레이어와 통계 정보는 원본을 유지"**하는 전략이다. 이를 통해 일반 소비자용 GPU(예: RTX 3090/4090 24GB)에서도 30B급 대형 모델을 구동할 수 있게 된다.


**1. 4비트 양자화 시 메모리 변화**

30B 파라미터 모델을 기준으로 계산하면 다음과 같은 변화가 발생한다.

- **BF16 (기본):** 파라미터당 2바이트 $\rightarrow$ 약 **60GB** 필요
- **4-bit (양자화):** 파라미터당 0.5바이트 $\rightarrow$ 약 **15GB** 필요 (이론상 1/4 수준)

실제로는 양자화 과정에서 발생하는 스케일링 계수(Scaling Factor)와 메타데이터 때문에 약 **17~18GB** 정도의 VRAM을 사용하게 된다.

**2. 왜 모든 파라미터를 양자화하지 않는가?**

모델의 성능(Perplexity) 저하를 최소화하기 위해 **혼합 정밀도(Mixed Precision)** 방식을 사용한다.

- **양자화 대상 (Linear Layers):** 모델의 대부분을 차지하는 행렬 연산 가중치(Attention, MLP 레이어 등)는 4비트로 변환하여 용량을 줄인다.
- **양자화 제외 (Sensitive Layers):**
    - **Normalization 레이어:** LayerNorm 등은 수치 민감도가 매우 높아 원본 정밀도(FP32/BF16)를 유지한다.
    - **Embedding 레이어:** 텍스트를 벡터로 변환하는 첫 단계이므로 정밀도가 중요하다.
    - **LM Head:** 최종 출력층은 예측 정확도를 위해 보통 양자화하지 않는다.

**3. 주요 양자화 기법 (NF4)**

단순히 소수점을 자르는 것이 아니라, 데이터의 분포를 고려한 알고리즘을 사용한다. 가장 대표적인 것이 **NF4(NormalFloat 4)**이다.

- **특징:** 가중치가 정규분포를 따른다는 가정하에, 값이 몰려 있는 구간에는 촘촘하게, 값이 적은 구간에는 넓게 비트를 할당한다.
- **장점:** 일반적인 4비트 정수형(Int4)보다 정보 손실이 훨씬 적어 모델의 추론 능력을 잘 보존한다.

In [1]:
%pip install -Uqqq transformers datasets accelerate trl peft bitsandbytes hg_transfer wandb

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement hg_transfer (from versions: none)
ERROR: No matching distribution found for hg_transfer


In [2]:
!nvidia-smi # NVIDIA 계열 GPU 확인

'nvidia-smi'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.


In [3]:
from dotenv import load_dotenv
import os

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
HF_TOKEN = os.getenv('HF_TOKEN')

In [4]:
import os

HF_TOKEN = os.environ['HF_TOKEN']

## 데이터셋 로드
https://huggingface.co/datasets/capybaraOh/naver-economy-news2stock

In [6]:
from datasets import load_dataset

dataset = load_dataset('capybaraOh/naver-economy-news2stock', split = 'train')
print(len(dataset))
dataset

1000


Dataset({
    features: ['system', 'user', 'assistant'],
    num_rows: 1000
})

In [7]:
dataset[0]

{'system': "\n당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,\n특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.\n\n다음 출력지시사항을 지켜주세요.\n1. 뉴스와 종목간의 연관성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 연관성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.\n",
 'user': '추경호 중기 수출지원 총력 무역금융 40조 확대\n앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대

In [8]:
# HuggingFace Dataset을 학습 / 평가 분리 후 chat 메시지 포맷으로 변환

test_ratio = 0.2

train_data = []
test_data = []

data_indices = list(range(len(dataset)))
test_size = int(len(dataset) * test_ratio)

test_data_indices = data_indices[:test_size]
train_data_indices = data_indices[test_size:]


def format_data(data):
    return {
        'messages': [
            {
                'role': 'system',
                'content': data['system']
            },
            {
                'role': 'user',
                'content': data['user']
            },
            {
                'role': 'assistant',
                'content': data['assistant']
            }
        ]
    }


train_data = [
    format_data(dataset[i])
    for i in train_data_indices
]

test_data = [
    format_data(dataset[i])
    for i in test_data_indices
]

print(len(train_data))
print(len(test_data))

800
200


In [9]:
train_data[256]

{'messages': [{'role': 'system',
   'content': "\n당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,\n특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.\n\n다음 출력지시사항을 지켜주세요.\n1. 뉴스와 종목간의 연관성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 연관성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.\n"},
  {'role': 'user',
   'content': '국토부 세종시 부적격 당첨자 철저히 조사해 엄중 조치\n원희룡 계약취소 및 주택환수 형사고발까지 필요 조치 취할 것 원희룡 국토교통부 장관. 2022.7.6 뉴스1 © News1 김명섭 기자 서울 뉴스1 박승희 기자 국토교통부는 감사원 감사에서 세종시 무자격 특공 당첨자가 무더기로 적발된 것과 관련해 위법행위 여부를 철저히 조사해 위법성이 밝혀지는 경우 엄중히 조치할 것 이라고 6일 밝혔다. 감사원은 세종시 이전기관 종사자 주택 특별공급에 대한 감사를 진행한 결과 총 45건 76명 에 대한 감사 결과를 확정했다. Δ대상관리 부실 Δ확인서 부당 발급 Δ확인서 위조 Δ중복 당첨 Δ지도·감독 소홀 등 사유로 무자격자가 대거 당첨된 것으로 드러났다. 국토부는 감사원으로부터 결과를 통보 받았으며 이에 따른 

In [10]:
from datasets import Dataset

train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)

train_dataset[256]

{'messages': [{'role': 'system',
   'content': "\n당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,\n특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.\n\n다음 출력지시사항을 지켜주세요.\n1. 뉴스와 종목간의 연관성을 발견할 수 없다면:\n    - stock_related를 False로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n2. 뉴스와 종목간의 연관성을 발견했다면:\n    - stock_related를 True로 작성하세요.\n    - summary에 뉴스의 요약을 작성하세요.\n    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.\n    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.\n    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.\n"},
  {'role': 'user',
   'content': '국토부 세종시 부적격 당첨자 철저히 조사해 엄중 조치\n원희룡 계약취소 및 주택환수 형사고발까지 필요 조치 취할 것 원희룡 국토교통부 장관. 2022.7.6 뉴스1 © News1 김명섭 기자 서울 뉴스1 박승희 기자 국토교통부는 감사원 감사에서 세종시 무자격 특공 당첨자가 무더기로 적발된 것과 관련해 위법행위 여부를 철저히 조사해 위법성이 밝혀지는 경우 엄중히 조치할 것 이라고 6일 밝혔다. 감사원은 세종시 이전기관 종사자 주택 특별공급에 대한 감사를 진행한 결과 총 45건 76명 에 대한 감사 결과를 확정했다. Δ대상관리 부실 Δ확인서 부당 발급 Δ확인서 위조 Δ중복 당첨 Δ지도·감독 소홀 등 사유로 무자격자가 대거 당첨된 것으로 드러났다. 국토부는 감사원으로부터 결과를 통보 받았으며 이에 따른 

## BaseModel + Quantizaton-Config

`BitsAndBytesConfig`는 Hugging Face Transformers에서 대형 모델을 8비트 또는 4비트로 양자화(quantization)하여 메모리 사용량을 줄이고, 저사양 환경에서도 대형 모델을 사용할 수 있게 도와주는 설정 클래스이다.

**주요 파라미터 목록**

| 파라미터명                  | 설명                                                                                          | 예시 값            |
|-----------------------------|----------------------------------------------------------------------------------------------|-------------------|
| `load_in_8bit`              | 8비트 양자화 활성화 여부. True로 설정 시 8비트로 모델 로드.                                    | True, False       |
| `load_in_4bit`              | 4비트 양자화 활성화 여부. True로 설정 시 4비트로 모델 로드.                                    | True, False       |
| `bnb_4bit_quant_type`       | 4비트 양자화 타입. `nf4`(NormalFloat4, 기본값), `fp4` 중 선택.                                 | "nf4", "fp4"      |
| `bnb_4bit_compute_dtype`    | 연산에 사용할 데이터 타입. 보통 `torch.float16`, `torch.bfloat16`, `torch.float32` 중 선택.     | torch.bfloat16    |
| `bnb_4bit_use_double_quant` | 이중 양자화 사용 여부. True로 설정 시 추가 양자화로 메모리 절감 가능.                           | True, False       |
| `llm_int8_threshold`        | 8비트 양자화 시 threshold 지정. 값이 낮을수록 더 많은 파라미터가 8비트로 변환됨.                | 0.0 ~ 6.0         |
| `llm_int8_skip_modules`     | 양자화에서 제외할 모듈 리스트.                                                                | ["lm_head"]       |
| `bnb_4bit_quant_storage`    | 4비트 파라미터 저장에 사용할 타입. 기본값은 `torch.uint8`.                                    | torch.uint8       |


- **load_in_8bit**  
  8비트 양자화를 활성화하는 플래그이다. True로 설정 시 모델 파라미터를 8비트 정수로 변환하여 메모리 사용량을 약 75%까지 줄일 수 있다.

- **load_in_4bit**  
  4비트 양자화를 활성화하는 플래그이다. True로 설정 시 더욱 극적인 메모리 절감 효과를 볼 수 있다. 4비트 양자화는 QLoRA 등 최신 연구에서 자주 사용된다.

- **bnb_4bit_quant_type**  
  4비트 양자화 시 사용할 데이터 타입을 지정한다.  
  - `nf4`: NormalFloat4 (기본값, QLoRA에서 주로 사용)  
  - `fp4`: FP4 타입.

- **bnb_4bit_compute_dtype**  
  연산(Forward/Backward) 시 사용할 데이터 타입을 지정한다.  
  - `torch.float16`, `torch.bfloat16`, `torch.float32` 등이 있다.  
  - 16비트 타입을 사용하면 연산 속도가 빨라지고, 메모리 사용량도 줄일 수 있다.

- **bnb_4bit_use_double_quant**  
  이중 양자화(nested quantization)를 활성화하는 옵션이다. True로 설정 시 한 번 더 양자화를 적용하여 메모리 사용량을 추가로 절감할 수 있다. 메모리 부족 시 유용하다.

- **llm_int8_threshold**  
  8비트 양자화 시 threshold 값을 조정하여, threshold 이하의 weight만 8비트로 변환한다. 값이 낮을수록 더 많은 파라미터가 8비트로 변환된다.

- **llm_int8_skip_modules**  
  양자화에서 제외할 모듈(레이어) 리스트를 지정한다. 예를 들어, 출력 레이어(`lm_head`) 등은 양자화에서 제외할 수 있다.

- **bnb_4bit_quant_storage**  
  4비트 파라미터 저장에 사용할 데이터 타입을 지정한다. 기본값은 `torch.uint8`이다.


**활용 팁**

- **메모리가 부족하다면**: `bnb_4bit_use_double_quant=True`로 설정.
- **정밀도가 중요하다면**: `bnb_4bit_quant_type="nf4"`로 설정.
- **학습 속도가 중요하다면**: `bnb_4bit_compute_dtype`를 16비트(float16, bfloat16)로 설정.

- `BitsAndBytesConfig`는 4비트/8비트 양자화 옵션을 통합 관리하며, 파라미터 조합을 통해 다양한 하드웨어 환경에 맞는 최적화가 가능하다.

In [11]:
# 4bit 양자화 설정(BitsAndBytesConfig)
from transformers import BitsAndBytesConfig # 양자화 설정 클래스
import torch

quant_config = BitsAndBytesConfig( 
    load_in_4bit = True,                     # 4bit 양자화(모델 가중치를 4bit)
    bnb_4bit_quant_type = 'nf4',             # 4bit 양자화 방식
    bnb_4bit_use_double_quant = True,        # 이중 양자화 : 메모리 / 정확도 균형 개선
    bnb_4bit_compute_dtype = torch.bfloat16  # 연산 방식 : bfloat16
)

## NCSOFT/Llama-VARCO-8B-Instruct란?
https://huggingface.co/NCSOFT/Llama-VARCO-8B-Instruct


* **기반 모델:** Meta의 Llama-3.1-8B 모델을 기반으로 한다.
* **개발 목적:** 한국어 능력을 극대화하는 동시에 영어 구사 능력도 유지하도록 설계되었다.
* **학습 방법:** 한국어와 영어 데이터셋을 활용한 지속 사전 학습(Continual Pre-training)을 거쳤으며, 이후 지도 미세 조정(SFT)과 직접 선호도 최적화(DPO)를 통해 인간의 선호도에 맞게 정렬되었다.


**SFT에서 한국어능력향상과 동시에 영어능력유지란:**

일반적으로 한국어 데이터를 대량으로 추가 학습시키면 기존에 모델이 가지고 있던 영어 지식이 손상되는 '파괴적 망각(Catastrophic Forgetting)' 현상이 발생한다. 엔씨소프트는 이를 방지하기 위해 **지속 사전 학습(Continual Pre-training)**을 적용했다.

**_1. 데이터 믹스(Data Mixing) 전략:_**

단순히 한국어 데이터만 밀어 넣는 것이 아니라, 모델이 이미 학습했던 영어 데이터와 고품질의 한국어 데이터를 특정 비율로 섞어 학습한다. 이를 통해 기존의 영어 추론 능력을 '복습'하면서 새로운 언어 체계를 '습득'하게 된다.

**_2. 토크나이저 효율화와 임베딩 확장:_**

기존 Llama-3.1의 토크나이저 성능을 유지하면서 한국어 표현력을 높이기 위해 어휘 사전(Vocabulary)을 최적화한다. 영어 토큰 정보는 건드리지 않고 한국어 토큰의 밀도를 높여 두 언어 간의 연결 고리를 강화하는 방식이다.

**_3. 지식 전이(Knowledge Transfer):_**

영어 데이터로 학습된 모델의 강력한 논리적 사고 능력을 한국어로 전이시키는 과정을 거친다.

* **추론 능력 유지:** 수학이나 코딩 같은 논리적 작업은 영어 데이터에서 배운 구조를 그대로 활용한다.
* **언어 정렬:** SFT(지도 미세 조정) 단계에서 동일한 질문을 한국어와 영어로 번급하며 학습시켜, 언어에 상관없이 일관된 답변을 내놓도록 유도한다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

pretrained_model_name = 'NCSOFT/Llama-VARCO-8B-Instruct'

model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name,
    dtype = torch.bfloat16,
    device_map = 'auto',
    quantization_config = quant_config
)

tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name)

In [ ]:
# 하나의 샘플만 openai 방식 메시지 -> llama3 방식 메시지로 변환
text = tokenizer.apply_chat_template(train_dataset[256]['messages'], tokenize = False)
print(text) # 나중에 runpod에서 실행 예정

NameError: name 'tokenizer' is not defined

In [ ]:
# labels에서 -100 제거한 후, assistant 정답 구간만 디코딩
label_ids = [token_id for token_id in batch['labels'][0].tolist() if token_id != -100]
text = tokenizer.decode(label_ids) # 문자열로 디코딩
text 

In [ ]:
# input_ids를 토큰/문자 단위로 디코딩해서 확인
tokens = tokenizer.convert_ids_to_tokens(batch['input_ids'][0].tolist())

text_tokens = [] # 토큰 ID를 디코딩한 문자열을 담을 리스트
for i, token_id in enumerate(batch['input_ids'][0].tolist()): # 토큰 ID 순회 
    decoded_str = tokenizer.decode([token_id]) # 토큰 1개를 문자열로 디코딩
    text_tokens.append(decoded_str) # 디코딩 결과 누적

In [ ]:
# 데이터프레임 시각화
import pandas as pd

df = pd.DataFrame({
    'token': text_tokens, 
    'input_ids': batch['input_ids'][0].tolist(),
    'attention_mask': batch['attention_mask'][0].tolist(),
    'labels': batch['labels'][0].tolist()
}).transpose()

pd.set_option('display.max_columns', None)
df

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfug(
    r = 8,
    lora_alpha = 32,
    lora_dropout = 0.1,
    bias = "none",
    target_modules = ['q_proj', 'v_proj'],
    task_type = "CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters

In [ ]:
#SFT(지도 미세조정) 학습 설정
from trl import SFTConfig  # TRL SFT 학습 설정 클래스

hub_model_id = 'capybaraOh/Llama-VARCO-8b-news2stock-analyzer'  # 학습 완료 후 업로드할 Hub 모델 ID

sft_config = SFTConfig(  # SFT 학습 하이퍼파라미터/저장/로그 설정
    output_dir="Llama-VARCO-8b-news2stock-analyzer", # 학습 완료된 모델과 체크포인트가 저장될 경로이다.
    num_train_epochs=3,                              # 전체 데이터셋을 반복 학습할 횟수(Epoch)이다.
    per_device_train_batch_size=2,                   # 각 GPU(장치)당 한 번에 처리할 데이터 샘플의 개수이다.
    gradient_accumulation_steps=2,                   # 그래디언트를 2번 누적한 후 가중치를 업데이트한다. (실제 배치 크기 = 2 * 2 = 4 효과를 낸다.)
    gradient_checkpointing=True,                     # VRAM 절약을 위해 중간 활성화 값을 저장하지 않고 역전파 시 재계산하는 설정이다.
    optim="adamw_torch_fused",                       # 최적화 알고리즘 설정이다. fused 버전은 CUDA에서 더 빠르다.
    logging_steps=10,                                # 10 스텝마다 학습 로그(Loss 등)를 출력한다.
    save_strategy="steps",                           # 체크포인트 저장 기준을 'steps'(스텝 수)로 설정한다. (옵션: 'epoch')
    save_steps=50,                                   # 50 스텝마다 모델 체크포인트를 저장한다.
    bf16=True,                                       # BF16(Brain Float 16) 정밀도를 사용하여 메모리를 아끼고 연산 속도를 높인다. (Ampere GPU 이상 권장)
    learning_rate=1e-4,                              # 학습률(Learning Rate)이다. 가중치 업데이트의 크기를 결정한다.
    max_grad_norm=0.3,                               # 그래디언트 클리핑 임계값이다. 그래디언트 폭주를 막아 학습 안정성을 높인다.
    warmup_steps=0.03,                               # 전체 학습 단계의 3% 동안 학습률을 서서히 올리는 웜업(Warmup)을 수행한다.
    lr_scheduler_type="constant_with_warmup",        # 학습률 스케줄러 타입이다. 여기서는 학습률을 서서히 올려준다.
    push_to_hub=True,                                # 학습이 끝나면 Hugging Face Hub에 모델을 자동으로 업로드한다.
    hub_model_id=hub_model_id,                       # Hub에 업로드될 때 사용될 저장소(Repository) ID이다.
    hub_token=True,                                  # Hub 업로드를 위해 인증 토큰을 사용한다.
    remove_unused_columns=False,                     # 데이터셋에서 모델의 forward 메서드 시그니처에 없는 컬럼을 자동으로 삭제하지 않도록 한다.
    dataset_kwargs={"skip_prepare_dataset": True},   # 데이터셋 처리 과정(packing 등)을 건너뛰도록 하는 설정이다.
    report_to=['wandb'],                                    # 학습 기록을 전송할 툴(WandB, Tensorboard 등)을 지정한다. 빈 리스트는 기록하지 않음을 의미한다.
    label_names=["labels"],                          # 손실(Loss) 계산 시 정답(Target)으로 사용할 데이터셋의 컬럼 이름이다.                              # chunked_nll 비활성화한다.
)

### 평가

In [ ]:
# 테스트셋 messages에서 프롬프트/정답(assistant) 텍스트 분리
prompt_list = []  # 프롬프트 (assistant 답변 내용 이전) 리스트
label_list = []   # 정답(assistant 답변 내용) 리스트

for messages in test_dataset["messages"]:
    # 채팅 템플릿 문자열로 반환
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    # assistant 답변 내용 전까지 input
    input = text.split('<|start_header_id|>assistant<|end_header_id|>\n')[0] + '<|start_header_id|>assistant<|end_header_id|>\n'
    # assistant 답변 내용 (종료 토큰 전) 추출
    labels = text.split('<|start_header_id|>assistant<|end_header_id|>\n')[1].split('<|eot_id|>')[0]
    prompt_list.append(input)
    label_list.append(labels)

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer, pipeline  # 토크나이저 자동 로더 / 파이프라인 생성
import torch

peft_model_name = hub_model_id  # 업로드 된 PEFT 모델 repo

# 파인튜닝 된 PEFT 모델 로드
finetuned_model = AutoPeftModelForCausalLM.from_pretrained(
    peft_model_name,
    dtype = torch.bfloat16,  # 가중치 로딩 dtype(bf16)
    device_map = 'auto'      # 환경에 맞춰 CPU / GPU 자동 배치
)

tokenizer = AutoTokenizer.from_pretrained(peft_model_name)  # 같은 repo에서 토크나이저 로드
# 텍스트 생성 파이프라인
pipe = pipeline('text-generation', model=finetuned_model, tokenizer=tokenizer)
pipe

In [ ]:
# <|eot_id|>의 토큰 id 추출
eos_token = tokenizer('<|eot_id|>', add_special_tokens=False)['input_ids'][0]
eos_token

In [ ]:
# 테스트 추론 함수 : 프롬프트, 정답, 모델응답 3개 샘플 비교 출력
def test_inference(pipe, prompt):
    # 파이프라인으로 결정론적인 답변 생성
    outputs = pipe(prompt, max_new_tokens=1024, eos_token_id=eos_token, do_sample=False)
    assistant_start = len(prompt)  # 입력 구간 이후 생성 결과만 사용
    return outputs[0]['generated_text'][assistant_start:].strip()  # 프롬프트 이후(생성된 부분)만 반환

for prompt, label in zip(prompt_list[10:13], label_list[10:13]):  # 10 ~ 12 샘플
    print(f"[prompt] : {prompt}")
    print(f"[label] : {label}")
    print(f"[response] : {test_inference(pipe, prompt)}")
    print('='*100)

### 추론 모델 사전 병합
- 진행 후에는 PEFT없이 해당 repo 모델로 바로 로드/추론 가능하다

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer, pipeline

merged_model_id = 'capybaraOh/Llama-VARCO-8b-news2stock-analyzer-4bit-merged'

tokenizer = AutoTokenizer.from_pretrained(peft_model_name)
# LoRA 어댑터를 포함한 모델 로드

finetuned_model = AutoPeftModelForCausalLM.from_pretrained(
    peft_model_name,
    dtype = torch.bfloat16, # 가중치 로딩 dtype(bf16)
    device_map = 'auto',    # 환경에 맞춰 CPU / GPU 자동 배치  
    quantization_config = quant_config # 4bit 양자화 설정 적용
)
merged_model = finetuned_model.merge_and_upload() # LoRA 가중치를 베이스 모델에 병합

merged_model.push_to_hub(merged_model_id, token = True) # 병합 모델 Hub 업로드
tokenizer.push_to_hub(merged_model_id, token = True)    # 토크나이저 Hub 업로드

In [ ]:
# GPU 메모리 해제(객체 삭제 + GC + CUDA 캐시 비우기)
del finetuned_model, pipe, merged_model

import gc     # 가비지 컬렉션 모듈
gc.collect()  # 참조가 끊긴 객체 메모리 정리

# torch.cuda.empty_cache() # CUDA 캐시 메모리 비움

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
model_id = 'capybaraOh/Llama-VARCO-8b-news2stock-analyzer-4bit-merged'

finetuned_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype = torch.bfloat16,
    device_map = 'auto'
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
pipe = pipeline('text-generation', model = finetuned_model, tokenizer = tokenizer)
pipe

In [ ]:
for prompt, label in zip(prompt_list[10:13], label_list[10:13]):  # 10 ~ 12 샘플
    print(f"[prompt] : {prompt}")
    print(f"[label] : {label}")
    print(f"[response] : {test_inference(pipe, prompt)}")
    print('='*100)

In [ ]:
# 뉴스 1건 입력받아 추론하는 함수
def inference(news):
    messages = [
        {'role': 'system', 'content': '''
당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,
특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.

다음 출력지시사항을 지켜주세요.
1. 뉴스와 종목간의 연관성을 발견할 수 없다면:
    - stock_related를 False로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
2. 뉴스와 종목간의 연관성을 발견했다면:
    - stock_related를 True로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.
    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.
    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.
'''},
        {'role': 'user', 'content': news}
    ]

    # messages를 채팅 프롬프트 문자열로 반환
    prompt = tokenizer.apply_chat_template(messages, tokenize=False)
    outputs = pipe(prompt, max_new_tokens=1024, eos_token_id=eos_token, do_sample=False)
    assistant_start = len(prompt)  # 입력 구간 이후 생성 결과만 사용
    return outputs[0]['generated_text'][assistant_start:].strip()  # 프롬프트 이후(생성된 부분)만 반환

In [ ]:
# 뉴스 1건 입력받아 추론하는 함수
def inference(news):
    messages = [
        {'role': 'system', 'content': '''
당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,
특정 상장 종목에 미치는 긍정/부정 영향여부, 이유, 근거를 분석하는 금융/경제 분석 전문가입니다.

다음 출력지시사항을 지켜주세요.
1. 뉴스와 종목간의 연관성을 발견할 수 없다면:
    - stock_related를 False로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
2. 뉴스와 종목간의 연관성을 발견했다면:
    - stock_related를 True로 작성하세요.
    - summary에 뉴스의 요약을 작성하세요.
    - 긍정영향이 예상되는 종목이 있다면, positive_stocks, positive_keywords, positive_reasons를 작성하세요.
    - 부정영향이 예상되는 종목이 있다면, negative_stocks, negative_keywords, negative_reasons를 작성하세요.
    - 값이 없는 경우 빈 문자열(''), 빈 리스트([])로 작성하세요.
'''},
        {'role': 'user', 'content': news}
    ]

    # messages를 채팅 프롬프트 문자열로 반환
    prompt = tokenizer.apply_chat_template(messages, tokenize=False)
    outputs = pipe(prompt, max_new_tokens=1024, eos_token_id=eos_token, do_sample=False)
    assistant_start = len(prompt)  # 입력 구간 이후 생성 결과만 사용
    return outputs[0]['generated_text'][assistant_start:].strip()  # 프롬프트 이후(생성된 부분)만 반환

In [ ]:
news = '''
현대건설-TS, 배송·주차로봇 다닐 아파트 안전 챙긴다

공동주택 내 미래모빌리티 대응 실증 MOU
단지 내 DRT·배송·서비스로봇 고려한 설계 연구
현대건설이 한국교통안전공단(TS)과 함께 공동주택 내 배송 로봇과 개인형 이동 수단 등 미래 모빌리티 도입 확대에 맞춘 안전기술을 개발하고 설계기준을 수립한다. 

현대건설은 지난 7일 웨스턴조선 서울에서 TS와 공동주택 미래 모빌리티 대응 안전기술 실증 협력을 위한 업무협약(MOU)을 체결했다고 8일 밝혔다. 이날 협약식에는 현대건설과 현대엔지니어링의 통합 R&D 조직인 HMG건설기술연구원의 김재영 원장과 송명준 인프라도시연구실장, 한국교통안전공단 민승기 모빌리티본부장 등이 참석했다.

현대건설에 따르면 이번 협약은 DRT(Demand Responsive Transport, 수요응답형교통)와 배송·서비스로봇, 퍼스널 모빌리티 등 공동주택에 다양한 이동 수단과 서비스 도입 확산을 계기로 이뤄졌다. 이에 맞춰 입주민의 보행 안전을 확보하고, 변화하는 교통 여건에 대응할 수 있는 단지 설계·운영기술을 개발하기 위해서라는 설명이다.

현대건설과 TS는 이번 협약을 통해 △공동주택 내 미래 모빌리티 안전성 증진 방안 공동 연구 △미래 모빌리티 도입에 따른 교통안전 개선사항 도출 △소프트웨어 중심 도로(Software Defined Road, SDR) 등의 기술을 활용한 도로환경 개선방안 검토 등을 추진한다. SDR은 소프트웨어로 도로 인프라를 제어·관리하고 다양한 이동 수단과 실시간으로 정보를 연계하는 차세대 도로 운영 기술이다.

양 기관은 DRT를 비롯해 배송·서비스로봇과 개인형 이동 수단 등 향후 주거단지에서 활용할 수 있는 모빌리티를 연구 대상에 포함해 안전한 이동환경을 구축한다는 계획이다. 단지 내 도로와 보행동선, 승하차·대기공간, 이동 수단별 운행체계 등 실제 주거환경을 구성하는 요소를 종합적으로 분석한 뒤 현장 실증을 통해 기술의 적용 가능성을 검증한다.

현대건설은 연구 결과를 토대로 입주민의 보행을 중심으로 단지 내 교통체계를 고도화한다. 모빌리티 운행 및 공간 이용 데이터를 활용해 도로와 동선 설계 최적화에 나선다는 것이다. 앞으로는 공동주택 기획·설계 단계부터 이동 수단별 운행 특성을 반영한 설계·운영 기준을 정립하고 미래 주거단지와 스마트시티를 연결하는 모빌리티 인프라 모델까지 발굴한다.

다양한 모빌리티 서비스가 주거단지 안으로 들어오면서 보행자와 이동 수단이 공존할 수 있는 공간계획과 운영체계가 중요하다는 게 현대건설의 설명이다. 특히 현대건설은 올해 압구정아파트지구 특별계획구역 재건축 사업 수주 과정에서 DRT를 비롯해 현대차그룹의 소형 모빌리티 '모베드(MobED)'를 비롯한 다양한 로봇과의 입주를 강조하기도 했다.▷관련기사: [압구정5 재건축]현대건설 승부수는 로봇, 그리고 갤러리아(2026년 5월21일)▷관련기사: "로봇과 함께 입주" 외친 현대건설의 '압구정 현대' 들여다보니…(2026년 5월12일)

현대건설 관계자는 "현대자동차·현대위아와 함께 공동주택 주차로봇 실증사업도 추진하고 있다"면서 "앞으로 주차로봇의 배치 및 운영기준을 정립하고 이를 단지 공간 계획과 연계한 스마트 주차 솔루션으로 발전시킬 것"이라고 말했다.
'''

inference(news)

### Base 모델과 비교

In [ ]:
from transformers import AutoModelForCausalLM, pipeline

base_model_id = 'NCSOFT/Llama-VARCO-8B-Instruct'  # 사전학습 모델명

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    dtype = torch.bfloat16,  # 가중치 로딩 dtype(bf16)
    device_map = 'auto'      # 환경에 맞춰 CPU / GPU 자동 배치
)

base_pipe = pipeline('text-generation', model=base_model, tokenizer=tokenizer)

# 베이스 모델과 LoRA 파인튜닝 모델의 응답과 정답 비교
for idx, (prompt, label) in enumerate(zip(prompt_list[10:13], label_list[10:13])):
    print(f"[샘플 {idx + 1}]")
    base_resp = test_inference(base_pipe, prompt)
    lora_resp = test_inference(pipe, prompt)
    print(f"[Base - 파인튜닝 전] {base_resp}")
    print(f"[LoRA - 파인튜닝 후] {lora_resp}")
    print(f"[Label] {label}")
    print("=" * 100)